# Initialization

In [0]:
import pyspark.sql.functions as F

In [0]:
ERP_CUSTOMER_RENAME_MAP = {
    "cid": "customer_id",
    "bdate": "birth_date",
    "gen": "gender"
}

# Reading From bronze table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

# Transformations

## Renaming columns

In [0]:
for old_name, new_name in ERP_CUSTOMER_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Remove the prefix "NAS"

In [0]:
df = df.withColumn(
    "customer_id",
    F.when(
        F.col("customer_id").startswith("NAS"),
        F.substring("customer_id", 4, 100)
    )
    .otherwise(F.col("customer_id"))
)

## Birth Date Validation

In [0]:
df = df.withColumn(
    "birth_date",
    F.when(
        F.col("birth_date") > F.current_date(),
        None
    )
    .otherwise(F.col("birth_date"))
)

## Gender Standardization

In [0]:
df = df.withColumn(
    "gender",
    F.when(
        F.upper(F.trim(F.col("gender"))).isin("M", "MALE"),
        "Male"
    )
    .when(
        F.upper(F.trim(F.col("gender"))).isin("F", "FEMALE"),
        "Female"
    )
    .otherwise("Unknown")
)

## Selecting Final Columns

In [0]:
df = df.select(
    "customer_id",
    "birth_date",
    "gender"
)

# Writing into Silver table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.erp_customers_info")
)